# AquaPINN Sequim Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwq2016/AquaPINN_tmp/blob/main/Examples/Colab/01_Sequim_Quickstart.ipynb)

Run the supplied Sequim example in four steps: set up the notebook, choose inputs and tracking settings, run the existing Sequim launcher, and review the results.


## 1. Set up

Run once. Colab saves results to Google Drive; local Jupyter saves them in `AquaPINN_Colab_Runs`.


In [ ]:
%pip -q install "numpy<2.5" scipy pandas matplotlib utm networkx scikit-learn tqdm six tomli-w

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/cwq2016/AquaPINN_tmp.git"
REPO_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "AquaPINN_Tracker3D").is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/AquaPINN_tmp")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                        REPO_URL, str(REPO_DIR)], check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_PARENT = Path("/content/drive/MyDrive/AquaPINN_Results")
except ImportError:
    OUTPUT_PARENT = REPO_DIR / "AquaPINN_Colab_Runs"
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
print("Ready. Results will be saved in:", OUTPUT_PARENT)


## 2. Choose inputs and tracking settings

Edit this block. Paths can point to files in Google Drive. TagID 2 is the bundled Sequim desktop example tag.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Input files
MAT_FILE = REPO_DIR / "Examples/Sequim/data/DriftingTags_timeCorrected.mat"
PHONE_FILE = REPO_DIR / "Examples/Sequim/config/sequim_testing_node_locs_w_beacons.csv"
TAG_FILE = REPO_DIR / "Examples/Sequim/config/DriftingTags.csv"
TEST_FILE = REPO_DIR / "Examples/Sequim/config/EA_test_description.csv"
GPS_FILE = REPO_DIR / "Examples/Sequim/config/drifting_GPS_day1_local_system.csv"

# Tracking settings
METHOD = "AML"  # AML, NN, or NNplus
NNPLUS_SAMPLES = 5
TIME_START = "2024-2-21 11:30:00"
TIME_END = "2024-2-21 13:30:00"
TIME_ZONE_HOURS = 0
TEMPERATURE_C = 8.0
HORIZONTAL_ONLY = True
DEVICE = "Automatic"  # Automatic or CPU only

display(pd.DataFrame({"Setting": ["Tag ID", "Method", "Time range", "Timezone", "Temperature (C)", "2D only", "Device"],
                        "Value": [2, METHOD, f"{TIME_START} to {TIME_END}", TIME_ZONE_HOURS, TEMPERATURE_C, HORIZONTAL_ONLY, DEVICE]}))


## 3. Run Sequim

This runs the project’s existing `main_Sequim.py` workflow. Sequim detections are already time-corrected, so no new synchronization step is needed.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, uuid
import tomllib, tomli_w

MAT_FILE = Path(MAT_FILE).expanduser().resolve()
PHONE_FILE = Path(PHONE_FILE).expanduser().resolve()
TAG_FILE = Path(TAG_FILE).expanduser().resolve()
TEST_FILE = Path(TEST_FILE).expanduser().resolve() if TEST_FILE else None
GPS_FILE = Path(GPS_FILE).expanduser().resolve() if GPS_FILE else None
SEQ_RUN = REPO_DIR / "Examples/Sequim/run"
PARAMS_FILE = SEQ_RUN / "params.toml"
with PARAMS_FILE.open("rb") as stream:
    params = tomllib.load(stream)

RUN_ID = uuid.uuid4().hex
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = OUTPUT_PARENT / f"Sequim_{stamp}_{RUN_ID[:8]}"
RUN_DIR.mkdir()
INPUTS = RUN_DIR / "inputs"
INPUTS.mkdir()
shutil.copy2(MAT_FILE, INPUTS / MAT_FILE.name)
RESULTS = RUN_DIR / "results"
RESULTS.mkdir()

params.update({
    "TagID": [2],
    "phoneFile": [str(PHONE_FILE)],
    "tagFile": [str(TAG_FILE)],
    "testFile": [str(TEST_FILE) if TEST_FILE else ""],
    "GPSFile": [str(GPS_FILE) if GPS_FILE else ""],
    "dataFolder": [str(INPUTS)],
    "output_folder": [str(RESULTS)],
    "time_start": [TIME_START], "time_end": [TIME_END],
    "timeZone": [TIME_ZONE_HOURS], "temperature_ref": [TEMPERATURE_C],
    "ignoreZ": ["True" if HORIZONTAL_ONLY else "False"],
    "trackMethod": ["AML" if METHOD == "AML" else "NN"],
    "nSamples": [1 if METHOD != "NNplus" else NNPLUS_SAMPLES],
})
effective_params = RUN_DIR / "params_effective.toml"
effective_params.write_text(tomli_w.dumps(params), encoding="utf-8")
(RUN_DIR / "params_effective.json").write_text(json.dumps(params, indent=2), encoding="utf-8")

track_rel = f"results/Track/track/track_Tag2_G726777EF_{METHOD}.csv"
run_record = {"schema": 1, "run_id": RUN_ID, "state": "starting",
              "method": METHOD, "track_csv": track_rel}
(RUN_DIR / "run.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")

# The existing launcher reads params.toml from its own folder; restore the example file afterward.
original_params = PARAMS_FILE.read_bytes()
PARAMS_FILE.write_text(tomli_w.dumps(params), encoding="utf-8")
env = os.environ.copy()
if DEVICE == "CPU only":
    env["CUDA_VISIBLE_DEVICES"] = ""
env["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + env.get("PYTHONPATH", "")
try:
    with (RUN_DIR / "console.log").open("w", encoding="utf-8") as log:
        result = subprocess.run([sys.executable, str(SEQ_RUN / "main_Sequim.py")],
                                cwd=SEQ_RUN, env=env, stdout=log,
                                stderr=subprocess.STDOUT, text=True)
finally:
    PARAMS_FILE.write_bytes(original_params)

TRACK_CSV = RUN_DIR / track_rel
if result.returncode != 0 or not TRACK_CSV.is_file():
    run_record["state"] = "failed"
    (RUN_DIR / "run.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")
    raise RuntimeError(f"Tracking failed. See {RUN_DIR / 'console.log'}")
run_record.update(state="completed", track_sha256=hashlib.sha256(TRACK_CSV.read_bytes()).hexdigest())
(RUN_DIR / "run.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")
print("Run complete:", RUN_DIR)


## 4. Review results

The run folder contains the track, plots, parameters, log, and desktop Simple Example run record. Keep the whole folder if you want to reopen the run later.


In [ ]:
import pandas as pd
from IPython.display import display, Image

track = pd.read_csv(TRACK_CSV)
display(track.head())

for figure in sorted(RESULTS.rglob("*.jpg"))[:4]:
    display(Image(filename=str(figure)))

print("Saved run folder:", RUN_DIR)
print("Desktop run record:", RUN_DIR / "run.json")
